# FMCW → range-Doppler → STAP/GMTI: exploration notebook

This notebook walks through the physics and processing chain of the `radar_stap` package. Every
computation calls the package modules (no re-implementation here), with a fixed seed, so the
figures are reproducible.

Physical picture: the radar transmits a linear chirp. An echo is a delayed copy; mixing it with the
transmitted chirp (*dechirping*) gives a **beat tone** whose frequency is proportional to range.
Across chirps the echo phase advances with the target's radial motion (**Doppler**). Across
antennas the phase changes with arrival angle. On a moving platform stationary ground clutter
also has Doppler, which depends on angle, so clutter fills a *ridge* in angle-Doppler space;
STAP filters jointly in space and time to cancel that ridge while keeping a target.

In [ ]:
import os, sys
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
try:
    import radar_stap  # installed with `pip install -e .`
except ImportError:  # running from a source checkout without installation
    sys.path.insert(0, os.path.abspath(os.path.join("..", "src")))
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

from radar_stap.config import load_simulation_config, sim_derived, describe_derived, wavelength_for
from radar_stap import waveforms as wf, plotting as pl
from radar_stap.simulator import simulate
from radar_stap.experiments import (process_cpi, detect_range_doppler, evaluate_rd_detections,
                                    run_stap_comparison, monte_carlo, truth_cell)
from radar_stap.range_processing import range_profile
from radar_stap.datacube import snapshots
from radar_stap.angle_processing import direction_from_angles
from radar_stap.covariance import sample_covariance
from radar_stap.response_basis import noise_covariance_scale
from radar_stap.stap import select_training_bins

# Default scenario with fewer clutter scatterers so the notebook runs quickly.
cfg = load_simulation_config("../configs/simulation.yaml",
                             {"clutter": {"num_scatterers": 1500}, "monte_carlo": {"num_trials": 6}})
d = sim_derived(cfg)
print(describe_derived(d))

## 1. FMCW waveform and the dechirped signal
Transmitted phase `phi(t) = 2π(f_c t + S t²/2)`; the IF after `tx·conj(rx)` mixing has phase
`2π(f_c τ + S t τ − S τ²/2)` — a tone at `f_b = S τ`. Below: the instantaneous frequency of the chirp
over the ADC window and the IF of a single point target (simulated by the package).

In [ ]:
single = load_simulation_config("../configs/simulation.yaml", {
    "clutter": {"enabled": False}, "stationary_reflectors": [], "stap": {"enabled": False},
    "targets": [{"name": "t", "range_m": 30.0, "angle_deg": 0.0, "snr_db": 20.0, "phase_deg": 0.0}]})
s1 = simulate(single)
t_f = wf.fast_time_axis(single.radar)
pl.plot_waveform(single.radar, t_f, s1.components["targets"][0, 0]);
x = s1.components["targets"][0, 0]
spec = np.abs(np.fft.fft(x * np.hanning(x.size), 16 * x.size))
fb_meas = np.argmax(spec) * single.radar.adc_sample_rate_hz / spec.size
print(f"measured beat {fb_meas/1e6:.4f} MHz, predicted S*2R/c = {wf.beat_frequency(30.0, 0.0, single.radar)/1e6:.4f} MHz")

## 2. Delay and Doppler
A receding target (`v_r = dR/dt > 0`) advances the slow-time phase by `2π (2 v_r/λ) T_c` per chirp
under the `tx_times_conj_rx` convention; the beat frequency also shifts by `2 v_r/λ`
(range-Doppler coupling, a few mm per m/s here).

In [ ]:
v = 4.0
mv = load_simulation_config("../configs/simulation.yaml", {
    "clutter": {"enabled": False}, "stationary_reflectors": [], "stap": {"enabled": False},
    "targets": [{"name": "t", "range_m": 30.0, "angle_deg": 0.0, "snr_db": 20.0, "radial_velocity_mps": v}]})
sm = simulate(mv)
X = np.fft.fft(sm.components["targets"][0] * np.hanning(mv.radar.num_adc_samples), axis=1)
k = np.argmax(np.abs(X[0]))
dphi = np.angle(X[1:, k] * np.conj(X[:-1, k]))
pred = np.angle(np.exp(2j*np.pi*(2*v/wavelength_for(mv.radar))*wf.chirp_period(mv.radar)))
print(f"per-chirp phase step: measured {dphi.mean():.4f} rad, predicted {pred:.4f} rad")
print(f"range-Doppler coupling: {d.range_doppler_coupling_m_per_mps*1e3:.2f} mm per m/s")

## 3. Full scene: range FFT and Doppler FFT
The default scene: side-looking platform at 8 m/s, 4-RX array, distributed ground clutter, two movers in
the look direction (20°), one fast mover elsewhere and a stationary discrete.

In [ ]:
sim = simulate(cfg)
proc = process_cpi(sim.data, cfg)
for t in sim.scene.truths:
    print(f"{t.name:14s} R={t.range_m:6.2f} m  v_r={t.radial_velocity_mps:+6.2f} m/s  angle={t.angle_deg:+6.1f} deg")
pl.plot_range_profile(proc.range_axis.range_m, range_profile(proc.Y), sim.scene.truths, proc.range_valid);

## 4. Range-Doppler map, clutter and CFAR
On a moving platform the clutter occupies |v_r| ≤ v_p; targets inside that band are not separable by
Doppler alone. Range-only CA-CFAR per Doppler column (the Doppler training cells next to the clutter
band would not be homogeneous).

In [ ]:
dets, cf = detect_range_doppler(proc, cfg)
ev = evaluate_rd_detections(dets, sim.scene.truths, proc, cfg)
v = proc.range_valid
pl.plot_range_doppler(proc.rd_map[v], proc.range_axis.range_m[v], proc.doppler_axis.velocity_mps, dets, sim.scene.truths)
for r in ev["targets"]:
    print(r["name"], "detected" if r["detected"] else "not detected", r.get("errors", ""))

## 5. Multichannel phase relationships
For the off-beam mover (exo-clutter, isolated in the RD map), the phase across the 4 RX channels follows
the far-field steering vector `a_m = exp(-j 2π x_m sin θ / λ)`.

In [ ]:
t = next(t for t in sim.scene.truths if t.name == "mover_offbeam")
rb, kb = truth_cell(t, cfg, proc)
z = proc.Z[:, int(round(kb)), int(round(rb))]
a = proc.model.spatial(direction_from_angles(t.angle_deg))
print("measured relative phase [deg]:", np.round(np.angle(z / z[0], deg=True), 1))
print("steering   relative phase [deg]:", np.round(np.angle(a / a[0], deg=True), 1))

## 6. Space-time data cube and steering vector
Snapshot for range bin r: `x[l*C + c] = V[c, l, r]` (pulse-major, channel fastest), so a single-TX steering
vector is `kron(b(ν), a(u))`. We check the simulated target snapshot against the model.

In [ ]:
X = snapshots(proc.V)
t = next(t for t in sim.scene.truths if t.name == "mover_fast")
rb, _ = truth_cell(t, cfg, proc)
from radar_stap.range_processing import range_fft
from radar_stap.datacube import to_virtual_cube
Yt, _ = range_fft(sim.components["targets"], cfg.radar, cfg.processing)
xt = snapshots(to_virtual_cube(Yt, cfg.array.tx_schedule))[int(round(rb))]
s = proc.model.steering(direction_from_angles(t.angle_deg), 2*t.radial_velocity_mps/proc.model.wavelength)
print("D =", X.shape[1], " |s^H x_t| / (|s||x_t|) =", abs(np.vdot(s, xt))/np.linalg.norm(s)/np.linalg.norm(xt))

## 7. STAP comparison and the covariance eigenvalue spectrum
`run_stap_comparison` applies every configured method to the *same* snapshots, training rule and Doppler
hypotheses. The clairvoyant covariance comes from the scatterer model; Brennan's rule
`N_ch + β(N_p − 1)` predicts the clutter rank.

In [ ]:
out = run_stap_comparison(sim, proc, cfg)
summ = out["summary"]
for name, m in summ["methods"].items():
    print(f"{name:12s} detections {m['num_detections']:3d}  targets", [(t['name'], t['detected']) for t in m['targets']])
a = out["sinr"][0]
pl.plot_eigenspectrum(a.clutter_eigs, a.sample_eigs, a.brennan_rank, noise_covariance_scale(cfg));

In [ ]:
setup = out["setup"]
mid = int(setup.test_bins[len(setup.test_bins)//2])
tr = select_training_bins(mid, cfg.stap.num_training, cfg.stap.num_guard, setup.valid_training)
pl.plot_angle_doppler(sample_covariance(setup.snaps[tr]), proc.model);

## 8. Conventional vs adaptive processing
SINR loss vs hypothesised velocity at the look angle (clairvoyant interference covariance), and the
angle-gated GMTI outputs with CFAR cells.

In [ ]:
pl.plot_sinr_loss(a, proc.model)
ra = proc.range_axis
pl.plot_gmti_maps(out["gmti"], lambda b: np.interp(np.asarray(b, float), np.arange(ra.num_bins), ra.range_m),
                  proc.model, setup.truth_cells);
for tname, s in summ["sinr_analysis"].items():
    print(tname, {k: round(v["output_sinr_db"], 1) for k, v in s["methods"].items()}, "optimum", round(s["optimal_sinr_db"], 1))

## 9–10. Detection metrics and limitations
A short Monte Carlo (few trials, for illustration only — use `scripts/simulate_scene.py` for the full
run) of P_D and P_FA with 95% Wilson intervals.

In [ ]:
mc = monte_carlo(sim, proc, cfg)
for name, m in mc.methods.items():
    print(f"{name:12s} Pd={m['pd']:.2f} {np.round(m['pd_95ci'],2)}  Pfa={m['pfa']:.1e} (design {m['pfa_design']:.0e})")

**Limitations of this experiment** (see README): point scatterers, no intrinsic clutter motion,
perfect channel calibration, Gaussian reflectivities, far-field processor model, oracle exclusion of
known targets from training, CA-CFAR thresholds that assume homogeneous exponential cells (adaptive
weights vary from bin to bin, so the measured P_FA can exceed the design value), and few Monte Carlo
trials here. Hardware behaviour is **not** validated by any of this.